## `fastplotlib` lorenz attractor demo

The Lorenz system is a dynamical system known for having chaotic solutions for certain parameter values and initial conditions. See [here](https://en.wikipedia.org/wiki/Lorenz_system) for more details. 

In [ ]:
import zmq
import numpy as np
import fastplotlib as fpl

## Setup zmq subscriber client

In [ ]:
context = zmq.Context()
sub = context.socket(zmq.SUB)
sub.setsockopt(zmq.SUBSCRIBE, b"")

# keep only the most recent message
sub.setsockopt(zmq.CONFLATE, 1)

# address must match publisher in Processor actor
sub.connect("tcp://127.0.0.1:5555")

In [ ]:
def get_buffer():
    """Gets the buffer from the publisher."""
    try:
        b = sub.recv(zmq.NOBLOCK)
    except zmq.Again:
        pass
    else:
        return b
    
    return None

## Create a figure

In [ ]:
# Create the figure
figure = fpl.Figure(
   cameras="3d",
   controller_types="fly",
)

# turn off axes
figure[0,0].axes.visible = False

In [ ]:
def update_frame(p):
    """Update the frame using data received from the socket."""
    buff = get_buffer()
    if buff is not None:
        # Deserialize the buffer into a NumPy array
        data = np.frombuffer(buff, dtype=np.float64)

        # Extract the frame number from the last index
        frame_num = int(data[-1]) 

        # after the first couple of frames generated, need to auto scale
        if frame_num == 3:
            p.auto_scale()

        data = data[:-1].reshape(-1, 3)

        # clear the plot to add updated data
        p.clear()
            
        # # add graphic for current data received
        p.add_line(data, cmap="jet", thickness=2)

        # Update the plot title with the frame number
        p.name = f"frame: {frame_num}"

## Use can use the `w, a, s, d` keys to "fly" around the plot as the data is being generated in 3D

In [ ]:
# Add the animation update function
figure[0, 0].add_animations(update_frame)

figure.show()